In [1]:
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_csv("adidas_usa.csv")

# Overview
print("Shape of Data:", df.shape)
print("\nColumns List:\n", df.columns.tolist())
df.head(3)

Shape of Data: (845, 21)

Columns List:
 ['index', 'url', 'name', 'sku', 'selling_price', 'original_price', 'currency', 'availability', 'color', 'category', 'source', 'source_website', 'breadcrumbs', 'description', 'brand', 'images', 'country', 'language', 'average_rating', 'reviews_count', 'crawled_at']


,index,url,name,sku,selling_price,original_price,currency,availability,color,category,...,source_website,breadcrumbs,description,brand,images,country,language,average_rating,reviews_count,crawled_at
0,0,https://www.adidas.com/us/beach-shorts/FJ5089....,Beach Shorts,FJ5089,40,NaN,USD,InStock,Black,Clothing,...,https://www.adidas.com,Women/Clothing,Splashing in the surf. Making memories with yo...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.5,35,2021-10-23 17:50:17.331255
1,1,https://www.adidas.com/us/five-ten-kestrel-lac...,Five Ten Kestrel Lace Mountain Bike Shoes,BC0770,150,NaN,USD,InStock,Grey,Shoes,...,https://www.adidas.com,Women/Shoes,Lace up and get after it. The Five Ten Kestrel...,adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.8,4,2021-10-23 17:50:17.423830
2,2,https://www.adidas.com/us/mexico-away-jersey/G...,Mexico Away Jersey,GC7946,70,NaN,USD,InStock,White,Clothing,...,https://www.adidas.com,Kids/Clothing,"Clean and crisp, this adidas Mexico Away Jerse...",adidas,"https://assets.adidas.com/images/w_600,f_auto,...",USA,en,4.9,42,2021-10-23 17:50:17.530834


In [4]:
# ==========================================
# STEP 2: DATA CLEANING & PREPROCESSING
# ==========================================

# 1. Missing Values & Duplicate Check
print("--- MISSING VALUES PER COLUMN ---")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# Drop duplicates if any
df = df.drop_duplicates().copy()

# 2. Price Cleaning (Removing currency symbols and typecasting)
price_cols = ['selling_price', 'original_price']

for col in price_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('$', '', regex=False)
        df[col] = df[col].str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows where selling_price is missing or invalid
if 'selling_price' in df.columns:
    df = df.dropna(subset=['selling_price'])

# 3. Clean string columns
text_cols = ['name', 'description', 'breadcrumbs', 'availability']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown').astype(str).str.strip()

print("\n--- CLEANED DATA SUMMARY ---")
print(f"Remaining Clean Rows: {df.shape[0]}")
df.info()

--- MISSING VALUES PER COLUMN ---
index              0
url                0
name               0
sku                0
selling_price      0
original_price    16
currency           0
availability       0
color              0
category           0
source             0
source_website     0
breadcrumbs        0
description        0
brand              0
images             0
country            0
language           0
average_rating     0
reviews_count      0
crawled_at         0
dtype: int64

Duplicate Rows: 0

--- CLEANED DATA SUMMARY ---
Remaining Clean Rows: 845
<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           845 non-null    int64  
 1   url             845 non-null    str    
 2   name            845 non-null    str    
 3   sku             845 non-null    str    
 4   selling_price   845 non-null    int64  
 5   original_price  829 non

In [5]:
# ==========================================
# STEP 3: FEATURE ENGINEERING & STOCKOUT METRICS
# ==========================================

# 1. Calculate Discount Percentage
if 'original_price' in df.columns and 'selling_price' in df.columns:
    df['discount_pct'] = np.where(
        df['original_price'] > 0,
        ((df['original_price'] - df['selling_price']) / df['original_price']) * 100,
        0
    )
    df['discount_pct'] = df['discount_pct'].clip(lower=0).round(2)

# 2. Stockout Detection
# Check availability status or size presence
if 'availability' in df.columns:
    df['is_out_of_stock'] = df['availability'].str.lower().str.contains('out of stock|discontinued|unavailable|no').astype(int)
elif 'breadcrumbs' in df.columns:
    df['is_out_of_stock'] = df['breadcrumbs'].str.lower().str.contains('out of stock').astype(int)
else:
    # Fallback default
    df['is_out_of_stock'] = 0

# 3. Calculate Potential Lost Revenue
df['potential_lost_revenue'] = np.where(df['is_out_of_stock'] == 1, df['selling_price'], 0)

# 4. Extract Category from Breadcrumbs or Name
if 'breadcrumbs' in df.columns:
    df['category'] = df['breadcrumbs'].apply(lambda x: str(x).split('/')[-1].strip() if '/' in str(x) else str(x))
else:
    df['category'] = 'General'

# 5. Summary Aggregations
category_summary = df.groupby('category').agg(
    total_products=('name', 'count'),
    out_of_stock_count=('is_out_of_stock', 'sum'),
    avg_price=('selling_price', 'mean'),
    total_lost_revenue=('potential_lost_revenue', 'sum')
).reset_index()

category_summary['stockout_rate_%'] = (category_summary['out_of_stock_count'] / category_summary['total_products'] * 100).round(2)
category_summary = category_summary.sort_values(by='total_lost_revenue', ascending=False)

print("--- OVERALL STOCKOUT METRICS ---")
print(f"Total Products Analyzed: {len(df)}")
print(f"Total Out-of-Stock Products: {df['is_out_of_stock'].sum()}")
print(f"Overall Stockout Rate: {(df['is_out_of_stock'].sum() / len(df) * 100):.2f}%")
print(f"Total Estimated Lost Revenue: ${df['potential_lost_revenue'].sum():,.2f}")

print("\n--- TOP CATEGORIES BY LOST REVENUE ---")
print(category_summary.head(10).to_string(index=False))

--- OVERALL STOCKOUT METRICS ---
Total Products Analyzed: 845
Total Out-of-Stock Products: 0
Overall Stockout Rate: 0.00%
Total Estimated Lost Revenue: $0.00

--- TOP CATEGORIES BY LOST REVENUE ---
   category  total_products  out_of_stock_count  avg_price  total_lost_revenue  stockout_rate_%
Accessories              82                   0  26.195122                   0              0.0
   Clothing             337                   0  39.332344                   0              0.0
      Shoes             426                   0  69.354460                   0              0.0
